In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Major_Dhyan_Chand_National_Stadium_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,350.0,NaN,224.0,104.0,152.0,NaN,73.0,74.0,74.0,139.0,350.0,277.0
1,2,353.0,263.0,89.0,105.0,198.0,167.0,93.0,90.0,74.0,148.0,296.0,290.0
2,3,337.0,210.0,103.0,114.0,219.0,185.0,87.0,75.0,54.0,131.0,387.0,264.0
3,4,378.0,290.0,157.0,160.0,262.0,179.0,43.0,67.0,50.0,141.0,385.0,160.0
4,5,358.0,251.0,97.0,139.0,281.0,199.0,59.0,59.0,48.0,132.0,372.0,128.0
5,6,325.0,179.0,167.0,165.0,221.0,253.0,40.0,63.0,79.0,148.0,347.0,176.0
6,7,351.0,156.0,155.0,165.0,249.0,213.0,NaN,61.0,46.0,125.0,392.0,218.0
7,8,363.0,173.0,114.0,158.0,197.0,208.0,34.0,61.0,126.0,159.0,388.0,274.0
8,9,369.0,151.0,110.0,247.0,187.0,198.0,68.0,62.0,116.0,150.0,376.0,205.0
9,10,305.0,295.0,226.0,173.0,215.0,187.0,121.0,67.0,96.0,120.0,358.0,260.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,350.000000,201.0,224.0,104.0,152.0,144.428571,73.000000,74.000000,74.000000,139.000000,350.000000,277.000000
1,2,353.000000,263.0,89.0,105.0,198.0,167.000000,93.000000,61.628571,74.000000,148.000000,296.000000,290.000000
2,3,337.000000,210.0,103.0,114.0,219.0,185.000000,87.000000,75.000000,54.000000,131.000000,387.000000,264.000000
3,4,378.000000,290.0,157.0,160.0,262.0,179.000000,43.000000,67.000000,50.000000,141.000000,385.000000,160.000000
4,5,358.000000,251.0,97.0,139.0,281.0,199.000000,59.000000,59.000000,48.000000,132.000000,372.000000,128.000000
5,6,325.000000,179.0,167.0,165.0,221.0,253.000000,40.000000,63.000000,79.000000,148.000000,347.000000,176.000000
6,7,351.000000,156.0,155.0,165.0,249.0,213.000000,73.685714,61.000000,46.000000,125.000000,392.000000,218.000000
7,8,363.000000,173.0,114.0,158.0,197.0,208.000000,73.685714,61.000000,126.000000,159.000000,388.000000,274.000000
8,9,369.000000,151.0,110.0,247.0,187.0,198.000000,68.000000,62.000000,116.000000,150.000000,376.000000,205.000000
9,10,305.000000,295.0,226.0,173.0,215.0,187.000000,73.685714,67.000000,96.000000,120.000000,358.000000,260.000000
